In [ ]:
!nvidia-smi

In [ ]:
%%writefile naive_softmax.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void naiveSoftmax(float *x, float *y, int rows, int cols) {
    int row = blockIdx.x*blockDim.x + threadIdx.x;
    if (row >= rows) return;
    float *xi = x + row*cols, *yi = y + row*cols;
    float sum = 0.f;
    for (int j = 0; j < cols; j++) sum += expf(xi[j]);
    for (int j = 0; j < cols; j++) yi[j] = expf(xi[j]) / sum;
}

int main() {
    int rows = 4096, cols = 1024;
    float *dx, *dy;
    cudaMalloc(&dx, rows*cols*4); cudaMalloc(&dy, rows*cols*4);
    int blk = 128, grd = (rows+blk-1)/blk;
    naiveSoftmax<<<grd,blk>>>(dx,dy,rows,cols); cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 100; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) naiveSoftmax<<<grd,blk>>>(dx,dy,rows,cols);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms,s,e);
    printf("Naive softmax  %.2f GB/s\n", 2.0*rows*cols*4*reps/(ms*1e6));
    cudaFree(dx); cudaFree(dy);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o naive_softmax naive_softmax.cu && ./naive_softmax